# Phase 6: Advanced Forecasting Models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Statistical
from prophet import Prophet

## 1. Deep Learning: LSTM (Long Short-Term Memory)
LSTMs require a 3D input tensor of shape `(samples, time_steps, features)`. We will reshape our sequential daily data for the top city (Delhi) to demonstrate the architecture.

In [ ]:
# Load base cleaned data (before Phase 4 engineered features, to let LSTM learn its own lags)
df = pd.read_csv('../data/processed/clean_air_quality.csv', parse_dates=['date'])
delhi_df = df[df['city'] == 'Delhi'].sort_values('date').reset_index(drop=True)

# We will use a few key features for the LSTM
features = ['pm2_5', 'pm10', 'no2', 'so2', 'co', 'o3', 'aqi']
data = delhi_df[features].fillna(delhi_df[features].median()).values

# Scale data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

# Create sequences (7 days lookback -> predict 1 day ahead)
TIME_STEPS = 7
X, y = [], []
for i in range(TIME_STEPS, len(scaled_data)):
    X.append(scaled_data[i-TIME_STEPS:i])
    # Index 6 corresponds to 'aqi'
    y.append(scaled_data[i, 6])
    
X, y = np.array(X), np.array(y)

# Train/Test Split (Chronological)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"LSTM Input shape: {X_train.shape}")

In [ ]:
# Define and train LSTM
model = Sequential([
    LSTM(50, activation='relu', input_shape=(TIME_STEPS, len(features))),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, verbose=1)


In [ ]:
# Predict and calculate RMSE
preds = model.predict(X_test)

# Inverse transform to get actual AQI values
# We need to create a dummy array matching the original feature count to inverse transform correctly
dummy_preds = np.zeros((len(preds), len(features)))
dummy_preds[:, 6] = preds.flatten()
inv_preds = scaler.inverse_transform(dummy_preds)[:, 6]

dummy_y = np.zeros((len(y_test), len(features)))
dummy_y[:, 6] = y_test
inv_y = scaler.inverse_transform(dummy_y)[:, 6]

lstm_rmse = np.sqrt(mean_squared_error(inv_y, inv_preds))
print(f"LSTM RMSE on Delhi Test Set: {lstm_rmse:.2f}")

## 2. Statistical Modeling: Prophet
Prophet expects a dataframe with two columns: `ds` (date) and `y` (target variable). We will fit Prophet on the same Delhi dataset to compare.

In [ ]:
prophet_df = delhi_df[['date', 'aqi']].rename(columns={'date': 'ds', 'aqi': 'y'})
prophet_df = prophet_df.dropna()

train_prophet = prophet_df.iloc[:split_idx]
test_prophet = prophet_df.iloc[split_idx:split_idx+len(y_test)]

m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.fit(train_prophet)

future = m.make_future_dataframe(periods=len(test_prophet))
forecast = m.predict(future)

prophet_preds = forecast['yhat'].iloc[-len(test_prophet):].values
prophet_rmse = np.sqrt(mean_squared_error(test_prophet['y'].values, prophet_preds))
print(f"Prophet RMSE on Delhi Test Set: {prophet_rmse:.2f}")

## 3. Advanced Models vs Baseline Conclusion

We now compare the advanced sequence models against our best Tabular model (LightGBM) which achieved an average RMSE of ~45 across all cities.

**Observations**:
- **LSTM**: While highly capable of capturing complex non-linear dynamics, LSTMs require extensive tuning (architecture size, epochs, dropout) to prevent overfitting on smaller datasets. 
- **Prophet**: Excellent at capturing pure seasonality (like the consistent winter pollution spikes in North India) without needing external features, though it struggles with random acute spikes.
- **Verdict**: The globally trained **LightGBM** model from Phase 5 remains highly competitive (and often superior) because it leverages data from *all* 26 cities simultaneously, learning macro-patterns that individual univariate city models (like Prophet) or data-hungry LSTMs miss on isolated datasets. LightGBM remains our chosen production model.